# Automated Machine Learning for Salary Prediction

## Project Overview

This project explores the use of Automated Machine Learning (AutoML) frameworks for predicting football players' salaries.

The objective is to build regression models that predict a player's `wage_eur` based on available player attributes and compare different AutoML approaches.

The project evaluates two frameworks:

- AutoGluon
- H2O AutoML



In [ ]:
%%capture
!pip install opendatasets

In [ ]:
import opendatasets as od
import pandas as pd

## 1. Data Loading and Preparation

The dataset contains information about football players from FIFA Career Mode datasets.

The target variable is:

**`wage_eur` — estimated player salary in euros.**

Observations with missing target values are removed before model training.

In [ ]:
df = pd.read_excel("Career Mode player datasets - FIFA 15-22.xlsx")
df = df.dropna(subset=["value_eur", "wage_eur"])
print(df.shape)
df.to_csv(df.to_csv("Career Mode player datasets - FIFA 15-22.csv"))

In [ ]:
df

,sofifa_id,player_url,short_name,long_name,player_positions,overall,potential,value_eur,wage_eur,age,...,lcb,cb,rcb,rb,gk,player_face_url,club_logo_url,club_flag_url,nation_logo_url,nation_flag_url
0,158023,https://sofifa.com/player/158023/lionel-messi/...,L. Messi,Lionel Andrés Messi Cuccittini,CF,93,95,100500000.0,550000.0,27,...,45+3,45+3,45+3,54+3,15+3,https://cdn.sofifa.net/players/158/023/15_120.png,https://cdn.sofifa.net/teams/241/60.png,https://cdn.sofifa.net/flags/es.png,https://cdn.sofifa.net/teams/1369/60.png,https://cdn.sofifa.net/flags/ar.png
1,20801,https://sofifa.com/player/20801/c-ronaldo-dos-...,Cristiano Ronaldo,Cristiano Ronaldo dos Santos Aveiro,"LW, LM",92,92,79000000.0,375000.0,29,...,52+3,52+3,52+3,57+3,16+3,https://cdn.sofifa.net/players/020/801/15_120.png,https://cdn.sofifa.net/teams/243/60.png,https://cdn.sofifa.net/flags/es.png,https://cdn.sofifa.net/teams/1354/60.png,https://cdn.sofifa.net/flags/pt.png
2,9014,https://sofifa.com/player/9014/arjen-robben/15...,A. Robben,Arjen Robben,"RM, LM, RW",90,90,54500000.0,275000.0,30,...,46+3,46+3,46+3,55+3,14+3,https://cdn.sofifa.net/players/009/014/15_120.png,https://cdn.sofifa.net/teams/21/60.png,https://cdn.sofifa.net/flags/de.png,https://cdn.sofifa.net/teams/105035/60.png,https://cdn.sofifa.net/flags/nl.png
3,41236,https://sofifa.com/player/41236/zlatan-ibrahim...,Z. Ibrahimović,Zlatan Ibrahimović,ST,90,90,52500000.0,275000.0,32,...,55+3,55+3,55+3,56+3,17+3,https://cdn.sofifa.net/players/041/236/15_120.png,https://cdn.sofifa.net/teams/73/60.png,https://cdn.sofifa.net/flags/fr.png,https://cdn.sofifa.net/teams/1363/60.png,https://cdn.sofifa.net/flags/se.png
4,167495,https://sofifa.com/player/167495/manuel-neuer/...,M. Neuer,Manuel Peter Neuer,GK,90,90,63500000.0,300000.0,28,...,38+3,38+3,38+3,36+3,87+3,https://cdn.sofifa.net/players/167/495/15_120.png,https://cdn.sofifa.net/teams/21/60.png,https://cdn.sofifa.net/flags/de.png,https://cdn.sofifa.net/teams/1337/60.png,https://cdn.sofifa.net/flags/de.png
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16149,222997,https://sofifa.com/player/222997/marcus-maier/...,M. Maier,Marcus Maier,CM,42,54,25000.0,2000.0,18,...,39,39,39,38,12,https://cdn.sofifa.net/players/222/997/15_120.png,https://cdn.sofifa.net/teams/111821/60.png,https://cdn.sofifa.net/flags/at.png,NaN,https://cdn.sofifa.net/flags/at.png
16150,220806,https://sofifa.com/player/220806/ellis-redman/...,E. Redman,Ellis Redman,CB,41,61,20000.0,2000.0,17,...,41,41,41,40,10,https://cdn.sofifa.net/players/220/806/15_120.png,https://cdn.sofifa.net/teams/112254/60.png,https://cdn.sofifa.net/flags/gb-eng.png,NaN,https://cdn.sofifa.net/flags/gb-wls.png
16151,225509,https://sofifa.com/player/225509/aaron-collins...,A. Collins,Aaron Graham John Collins,ST,41,50,30000.0,2000.0,17,...,31,31,31,32,14,https://cdn.sofifa.net/players/225/509/15_120.png,https://cdn.sofifa.net/teams/112254/60.png,https://cdn.sofifa.net/flags/gb-eng.png,NaN,https://cdn.sofifa.net/flags/gb-wls.png
16153,217591,https://sofifa.com/player/217591/piotr-zemlo/1...,P. Żemło,Piotr Żemło,"LM, LB",40,50,15000.0,2000.0,18,...,53-3,53-3,53-3,51-1,12,https://cdn.sofifa.net/players/217/591/15_120.png,https://cdn.sofifa.net/teams/1873/60.png,https://cdn.sofifa.net/flags/pl.png,NaN,https://cdn.sofifa.net/flags/pl.png


## 2. AutoGluon

AutoGluon is used to automatically train and compare multiple machine learning models for the regression task.

The dataset is split into training and test sets using an 80/20 split.

The primary optimization metric is RMSE.

In [ ]:
%%capture
!pip install autogluon

In [ ]:
from autogluon.tabular import TabularPredictor, TabularDataset
from sklearn.model_selection import train_test_split

data = TabularDataset("Career Mode player datasets - FIFA 15-22.csv")  # загрузка данных
train_data_1, test_data_1 = train_test_split(data, test_size=0.2, random_state=42)  # разделение на подвыборки

In [ ]:
predictor = TabularPredictor(
    label="wage_eur",
    eval_metric="rmse",
    problem_type="regression"
).fit(
    train_data_1,
    presets="medium_quality_faster_train",
    time_limit=180
)

No path specified. Models will be saved in: "AutogluonModels/ag-20251017_103841"
Preset alias specified: 'medium_quality_faster_train' maps to 'medium_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Oct  2 10:42:05 UTC 2025
CPU Count:          2
Memory Avail:       11.00 GB / 12.67 GB (86.8%)
Disk Space Avail:   62.03 GB / 107.72 GB (57.6%)
Presets specified: ['medium_quality_faster_train']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 180s
AutoGluon will save models to "/content/AutogluonModels/ag-20251017_103841"
Train Data Rows:    12675
Train Data Columns: 110
Label Column:       wage_eur
Problem Type:       regression
Preprocessing data ...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available M

[1000]	valid_set's rmse: 2921.06
[2000]	valid_set's rmse: 2897.2
[3000]	valid_set's rmse: 2893.85
[4000]	valid_set's rmse: 2893.72


	-2893.502	 = Validation score   (-root_mean_squared_error)
	60.59s	 = Training   runtime
	1.71s	 = Validation runtime
Fitting model: LightGBM ... Training model for up to 93.60s of the 93.59s of remaining time.
	Fitting with cpus=1, gpus=0, mem=0.1/10.7 GB
	-2206.8627	 = Validation score   (-root_mean_squared_error)
	12.55s	 = Training   runtime
	0.25s	 = Validation runtime
Fitting model: RandomForestMSE ... Training model for up to 80.57s of the 80.57s of remaining time.
	Fitting with cpus=2, gpus=0
	-3893.4895	 = Validation score   (-root_mean_squared_error)
	243.39s	 = Training   runtime
	0.19s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ... Training model for up to 156.81s of the -163.20s of remaining time.
	Ensemble Weights: {'LightGBM': 0.75, 'LightGBMXT': 0.25}
	-2106.7499	 = Validation score   (-root_mean_squared_error)
	0.01s	 = Training   runtime
	0.0s	 = Validation runtime
AutoGluon training complete, total runtime = 343.28s ... Best model: WeightedEnsemble_L2 

## Model Comparison

AutoGluon automatically trains multiple candidate models and ranks them according to their validation performance.

In [ ]:
leaderboard = predictor.leaderboard()
leaderboard

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-2106.749926,root_mean_squared_error,1.954703,73.159938,0.000607,0.011864,2,True,4
1,LightGBM,-2206.862675,root_mean_squared_error,0.246208,12.553726,0.246208,12.553726,1,True,2
2,LightGBMXT,-2893.502024,root_mean_squared_error,1.707889,60.594348,1.707889,60.594348,1,True,1
3,RandomForestMSE,-3893.489468,root_mean_squared_error,0.185450,243.394420,0.185450,243.394420,1,True,3


In [ ]:
predictor.evaluate(train_data_1)

{'root_mean_squared_error': np.float64(-839.8781707246244),
 'mean_squared_error': -705395.3416597412,
 'mean_absolute_error': -277.25218030579697,
 'r2': 0.998705622739418,
 'pearsonr': 0.999352878105842,
 'median_absolute_error': np.float64(-155.767578125)}

## Model Evaluation

The model performance is evaluated using several regression metrics:

- **R²** — proportion of variance explained by the model;
- **RMSE** — average prediction error with a stronger penalty for large errors;
- **MAE** — average absolute prediction error.

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

predictions = predictor.predict(train_data_1)

r2 = r2_score(train_data_1['wage_eur'], predictions)
rmse = np.sqrt(mean_squared_error(train_data_1['wage_eur'], predictions))
mae = mean_absolute_error(train_data_1['wage_eur'], predictions)

print(f"R² (коэффициент детерминации): {r2:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

R² (коэффициент детерминации): 0.9987
RMSE: 839.8782
MAE: 277.2522


In [ ]:
predictor.evaluate(test_data_1)

{'root_mean_squared_error': np.float64(-5465.91970560817),
 'mean_squared_error': -29876278.228155702,
 'mean_absolute_error': -1120.923486335829,
 'r2': 0.9515372347571525,
 'pearsonr': 0.9757440936953647,
 'median_absolute_error': np.float64(-357.67822265625)}

In [ ]:
predictions = predictor.predict(test_data_1)

r2 = r2_score(test_data_1['wage_eur'], predictions)
rmse = np.sqrt(mean_squared_error(test_data_1['wage_eur'], predictions))
mae = mean_absolute_error(test_data_1['wage_eur'], predictions)

print(f"R² (коэффициент детерминации): {r2:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

R² (коэффициент детерминации): 0.9515
RMSE: 5465.9197
MAE: 1120.9235


In [ ]:
y_true = test_data_1[["wage_eur"]]
y_true.columns = ["true_wage"]
test_data = test_data_1.drop(columns=["wage_eur"])

In [ ]:
y_pred = predictor.predict(test_data_1)

In [ ]:
result = pd.concat([y_true, y_pred], axis=1)
result["diff"] = (result["true_wage"]-result["wage_eur"]).abs()
result.sort_values(by=["diff"], ascending=False)

,true_wage,wage_eur,diff
0,550000.0,283409.968750,266590.031250
27,180000.0,237546.562500,57546.562500
33,160000.0,196317.937500,36317.937500
41,170000.0,202210.968750,32210.968750
19,160000.0,191857.687500,31857.687500
...,...,...,...
12614,2000.0,2001.416260,1.416260
12416,2000.0,2001.265625,1.265625
12849,2000.0,1999.053589,0.946411
10471,4000.0,4000.778076,0.778076


## Feature Importance

Feature importance analysis is performed to identify which player attributes contribute most to salary predictions.

In [ ]:
sample_data = train_data.sample(n=1000, random_state=42)

importance = predictor.feature_importance(data=sample_data)
print(importance)

These features in provided data are not utilized by the predictor and will be ignored: ['player_url', 'release_clause_eur', 'mentality_composure', 'st', 'rs', 'cf', 'rf', 'rw', 'cam', 'ram', 'cm', 'rcm', 'rm', 'cdm', 'rdm', 'rwb', 'cb', 'rcb', 'rb', 'player_face_url', 'club_logo_url', 'nation_flag_url']
Computing feature importance via permutation shuffling for 88 features using 995 rows with 5 shuffle sets...
	3038.55s	= Expected runtime (607.71s per shuffle set)
	2239.47s	= Actual runtime (Completed 5 of 5 shuffle sets)


                    importance    stddev   p_value  n  p99_high   p99_low
Unnamed: 0            0.105126  0.009309  0.000007  5  0.124294  0.085957
dob                   0.057487  0.004682  0.000005  5  0.067127  0.047848
club_name             0.030151  0.002010  0.000002  5  0.034289  0.026012
value_eur             0.008442  0.001826  0.000247  5  0.012201  0.004683
age                   0.008040  0.002010  0.000432  5  0.012179  0.003901
...                        ...       ...       ... ..       ...       ...
potential            -0.000603  0.000550  0.964758  5  0.000530 -0.001736
league_level         -0.000804  0.001101  0.911096  5  0.001463 -0.003071
movement_reactions   -0.001206  0.000449  0.998059  5 -0.000281 -0.002131
ldm                  -0.001206  0.001798  0.896000  5  0.002496 -0.004908
lam                  -0.001407  0.000899  0.987552  5  0.000444 -0.003258

[88 rows x 6 columns]


In [ ]:
from sklearn.metrics import classification_report
print("Детальный отчет по классификации (полный текст):")
print(classification_report(y_true, y_pred))

Детальный отчет по классификации (полный текст):
              precision    recall  f1-score   support

      2000.0       0.92      0.98      0.95       970
      3000.0       0.77      0.64      0.70       270
      4000.0       0.72      0.70      0.71       330
      5000.0       0.61      0.71      0.66       243
      6000.0       0.63      0.54      0.58       142
      7000.0       0.65      0.54      0.59       129
      8000.0       0.63      0.64      0.64       108
      9000.0       0.58      0.53      0.56       119
     10000.0       0.59      0.69      0.64       139
     15000.0       0.69      0.47      0.56       117
     20000.0       0.54      0.78      0.64       106
     25000.0       0.51      0.48      0.49        84
     30000.0       0.44      0.27      0.34        70
     35000.0       0.26      0.28      0.27        67
     40000.0       0.24      0.38      0.29        32
     45000.0       0.25      0.19      0.22        21
     50000.0       0.33      0.2

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# 3. H2O AutoML Comparison

To compare AutoML frameworks, the same salary prediction task is also solved using H2O AutoML.

The comparison focuses on:

- model performance;
- prediction accuracy;
- automated model selection;
- workflow differences between frameworks.

In [ ]:
%%capture
!pip install h2o

In [ ]:
import h2o
h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "11.0.28" 2025-07-15; OpenJDK Runtime Environment (build 11.0.28+6-post-Ubuntu-1ubuntu122.04.1); OpenJDK 64-Bit Server VM (build 11.0.28+6-post-Ubuntu-1ubuntu122.04.1, mixed mode, sharing)
  Starting server from /usr/local/lib/python3.12/dist-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmp99fakfp1
  JVM stdout: /tmp/tmp99fakfp1/h2o_unknownUser_started_from_python.out
  JVM stderr: /tmp/tmp99fakfp1/h2o_unknownUser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,04 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.8
H2O_cluster_version_age:,6 days
H2O_cluster_name:,H2O_from_python_unknownUser_r4juby
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.168 Gb
H2O_cluster_total_cores:,2
H2O_cluster_allowed_cores:,2
H2O_cluster_status:,"locked, healthy"


In [ ]:
data = h2o.import_file("Career Mode player datasets - FIFA 15-22.csv")
train_data, test_data = data.split_frame(ratios=[0.8])

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [ ]:
from h2o.automl import H2OAutoML
model = H2OAutoML(max_models=10, max_runtime_secs=300, seed=42)
model.train(y="wage_eur", training_frame=train_data)

AutoML progress: |
21:22:41.575: _train param, Dropping bad and constant columns: [player_url, mentality_composure, short_name, release_clause_eur, player_face_url, long_name]

██████
21:23:04.381: _train param, Dropping bad and constant columns: [player_url, mentality_composure, short_name, release_clause_eur, player_face_url, long_name]

█
21:23:10.376: _train param, Dropping bad and constant columns: [player_url, mentality_composure, short_name, release_clause_eur, player_face_url, long_name]

████
21:23:33.120: _train param, Dropping bad and constant columns: [player_url, mentality_composure, short_name, release_clause_eur, player_face_url, long_name]

███
21:23:43.443: _train param, Dropping bad and constant columns: [player_url, mentality_composure, short_name, release_clause_eur, player_face_url, long_name]

████
21:24:03.195: _train param, Dropping bad and constant columns: [player_url, mentality_composure, short_name, release_clause_eur, player_face_url, long_name]

██
21:24:1

Model Details
=============
H2OXGBoostEstimator : XGBoost
Model Key: XGBoost_3_AutoML_1_20251014_212240


Model Summary: 
    number_of_trees
--  -----------------
    60

ModelMetricsRegression: xgboost
** Reported on train data. **

MSE: 4323118.256149549
RMSE: 2079.2109696107195
MAE: 1244.8089131112567
RMSLE: NaN
Mean Residual Deviance: 4323118.256149549

ModelMetricsRegression: xgboost
** Reported on validation data. **

MSE: 63134193.94209165
RMSE: 7945.702860168612
MAE: 1965.0988400009678
RMSLE: NaN
Mean Residual Deviance: 63134193.94209165

Scoring History: 
    timestamp            duration    number_of_trees    training_rmse    training_mae    training_deviance    validation_rmse    validation_mae    validation_deviance
--  -------------------  ----------  -----------------  ---------------  --------------  -------------------  -----------------  ----------------  ---------------------
    2025-10-14 21:24:43  0.003 sec   0                  26550.4          13201.7         7.04922e+08          31532.3            14074.5           9.94283e+08
    2025-10-14 21:24:43  0.492 sec   5                  6468.24          2833.28         4.18381e+07          11469.7            3276.98           1.31553e+08
    2025-10-14 21:24:44  0.773 sec   10                 3732.71          1893.9          1.39332e+07          9061.22            2340.99           8.21057e+07
    2025-10-14 21:24:44  1.056 sec   15                 3224.7           1732.6          1.03987e+07          8445.9             2243.13           7.13333e+07
    2025-10-14 21:24:44  1.284 sec   20                 3032.27          1656.86         9.19465e+06          8170.41            2204.11           6.67556e+07
    2025-10-14 21:24:45  1.704 sec   25                 2805.25          1558.42         7.86941e+06          8033.97            2120.23           6.45447e+07
    2025-10-14 21:24:45  2.064 sec   30                 2656.71          1498.46         7.05809e+06          7990.19            2094.33           6.38431e+07
    2025-10-14 21:24:45  2.336 sec   35                 2559.52          1455.32         6.55116e+06          7958.95            2064.91           6.3345e+07
    2025-10-14 21:24:46  2.759 sec   40                 2450.41          1410.09         6.00449e+06          7985.14            2054.67           6.37624e+07
    2025-10-14 21:24:46  3.019 sec   45                 2303.82          1343.85         5.3076e+06           7906.98            2008.29           6.25203e+07
    2025-10-14 21:24:46  3.259 sec   50                 2226.92          1309.32         4.95918e+06          7933.97            1997.25           6.29479e+07
    2025-10-14 21:24:46  3.525 sec   55                 2155.76          1279.95         4.64731e+06          7969.16            1987.04           6.35075e+07
    2025-10-14 21:24:47  3.766 sec   60                 2079.21          1244.81         4.32312e+06          7945.7             1965.1            6.31342e+07

Variable Importances: 
variable                        relative_importance    scaled_importance       percentage
------------------------------  ---------------------  ----------------------  ----------------------
C1                              3493499240448.0        1.0                     0.4718050130245208
overall                         2453645557760.0        0.7023460973889747      0.3313704096263266
value_eur                       816539959296.0         0.2337312542799604      0.11027557746979431
gk                              234394877952.0         0.06709458391693872     0.031655561038806075
dob                             52878151680.0          0.015136156627078299    0.007141314574579865
lam                             51114848256.0          0.014631418167832528    0.006903176439241436
club_team_id                    34936401920.0          0.010000403468105469    0.004718240488519964
international_reputation        32080934912.0          0.00918303760898657     0.004332603178712573
age            

In [ ]:
y_true = test_data[["wage_eur"]]
test_data = test_data.drop("wage_eur")

In [ ]:
y_pred = model.leader.predict(test_data)

xgboost prediction progress: |███████████████████████████████████████████████████| (done) 100%


# Conclusions

This project demonstrates how AutoML can accelerate the development of machine learning models for tabular regression tasks.

Two AutoML frameworks were applied to predict football players' salaries:

- AutoGluon
- H2O AutoML

The analysis included automated model training, model comparison, regression evaluation, prediction error analysis, and feature importance.

The project highlights the practical trade-off between rapid automated experimentation and model interpretability.

In [ ]:
result = pd.concat([y_true.as_data_frame(), y_pred.as_data_frame()], axis=1)
result["diff"] = (result["wage_eur"]-result["predict"].round()).abs().round()
result.sort_values(by=["diff"], ascending=False)

/usr/local/lib/python3.12/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"
/usr/local/lib/python3.12/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


,wage_eur,predict,diff
27,80000,123757.742188,43758.0
43,45000,83790.898438,38791.0
17,100000,135161.875000,35162.0
16,100000,133748.109375,33748.0
171,30000,63352.503906,33353.0
...,...,...,...
1523,6000,5997.868652,2.0
1855,5000,4998.124512,2.0
2381,2000,2000.681885,1.0
2503,2000,2000.681885,1.0
